# 03 — Acompanhamento da acurácia

Este notebook simula o ciclo de avaliação: previsões são registradas antes dos realizados; quando as vendas ficam disponíveis, as métricas são atualizadas. Não há persistência, integração com PDV/estoque ou agendamento em produção nesta fase.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from demand_forecasting.backtesting import expanding_window_backtest
from demand_forecasting.monitoring import evaluate_forecast_log
from demand_forecasting.synthetic import generate_synthetic_demand

sns.set_theme(style="whitegrid")
panel = generate_synthetic_demand(random_state=42)
predictions = expanding_window_backtest(
    panel, horizon=7, min_train_days=365, step_days=56,
    n_estimators=50, random_state=42,
)
predictions.head()

## Previsões pendentes e resultados conhecidos

Um registro com `y_true` nulo representa uma previsão ainda sem resultado observado. O avaliador ignora esses registros até a chegada do realizado. No exemplo, ocultamos temporariamente o último dia previsto e depois o revelamos para demonstrar o recálculo.

In [ ]:
forecast_log = predictions.copy()
latest_target_date = forecast_log["date"].max()
pending_log = forecast_log.copy()
pending_mask = pending_log["date"] == latest_target_date
pending_log.loc[pending_mask, "y_true"] = float("nan")

print(f"Métricas antes dos realizados de {latest_target_date.date()}:")
display(evaluate_forecast_log(pending_log).round(3))

# Simula a chegada dos valores realizados para as previsões pendentes.
pending_log.loc[pending_mask, "y_true"] = forecast_log.loc[pending_mask, "y_true"].to_numpy()
print(f"Métricas após os realizados de {latest_target_date.date()}:")
display(evaluate_forecast_log(pending_log).round(3))

## Evolução das métricas após cada ciclo

O gráfico abaixo recalcula as métricas acumuladas à medida que cada data de teste é observada. Para operação, prefira também janelas móveis, cortes por produto/horizonte e comparação com baseline; métricas acumuladas podem mascarar degradação recente.

In [ ]:
history = []
for as_of_date in sorted(predictions["date"].unique()):
    known = predictions.loc[predictions["date"] <= as_of_date]
    snapshot = evaluate_forecast_log(known).assign(as_of_date=as_of_date)
    history.append(snapshot)
metrics_history = pd.concat(history, ignore_index=True)

sns.lineplot(data=metrics_history, x="as_of_date", y="WAPE", hue="model")
plt.title("WAPE acumulado conforme chegam os realizados")
plt.xlabel("Data de avaliação")
plt.ylabel("WAPE acumulado")
plt.show()

## Caminho para operação

1. Conectar uma fonte versionada de pedidos/vendas, estoque disponível, catálogo e calendário.
2. Salvar cada previsão com produto, data de geração, data-alvo, horizonte, versão do modelo e valor previsto. Não sobrescrever previsões históricas.
3. Associar o realizado após o fechamento da data, registrando rupturas e qualidade do dado.
4. Publicar métricas por produto e horizonte, comparação com baseline e alertas de degradação.
5. Definir limites com a operação e custos de ruptura/desperdício; WAPE/MAE não representam diretamente o custo de estoque.

Uma rotina contínua real exigirá armazenamento, agendamento, observabilidade e integração segura. A série sintética serve apenas para validar o fluxo de cálculo.